# 2. Turn the decoder into a bounded replay controller
 **Question:** how does command policy change what a decoder does?
 This is accelerated software replay of **validation run R08**, not a live BCI. No person receives contingent feedback, and no hardware is controlled.
 The helper uses sample order, a virtual arrival clock, a bounded past-window buffer, and a persistent causal filter. Annotation labels are read only after the policy decision for retrospective scoring.

In [ ]:
%pip -q install "numpy>=1.26,<3" "scipy>=1.11,<2" "mne>=1.8,<2" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4"

In [ ]:
from pathlib import Path
import hashlib, sys, urllib.request
SOURCE_COMMIT = "22dcdced86cc425a2d50ffa8516fe9581730ebc9"
EXPECTED_SHA256 = "90c8c015e114b3b9a89bb4206dbc4f6094969a792b1a0263e0009e673f935567"
candidates = [Path("bci_studio.py"), Path("chapters/ch20-bci-foundations/bci_studio.py")]
helper = next((p for p in candidates if p.is_file()), None)
if helper is None:
    target = Path("chapter20_runtime")
    target.mkdir(exist_ok=True)
    helper = target / "bci_studio.py"
    url = f"https://raw.githubusercontent.com/ZagElSayed/Introduction-to-Neuroinformatics/{SOURCE_COMMIT}/chapters/ch20-bci-foundations/bci_studio.py"
    payload = urllib.request.urlopen(url, timeout=60).read()
    if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
        raise RuntimeError("Downloaded helper checksum mismatch")
    helper.write_bytes(payload)
sys.path.insert(0, str(helper.resolve().parent))
print("Helper:", helper, "SHA-256:", hashlib.sha256(helper.read_bytes()).hexdigest())
import bci_studio as bci

In [ ]:
experiment = bci.build_experiment()
clean = bci.replay(experiment)  # R08 validation by default
print(clean["summary"])

## Inspect the decoder-to-command boundary
 The naive comparator executes the argmax on every valid feature window. The gated policy requires score ≥ .70, three consecutive matching updates, and a one-second refractory interval.
 These overlapping updates are correlated; this is not three independent observations. The score is not an empirically calibrated probability of intent.
 Both comparators share basic input QC and warm-up. Neither operates on invalid chunks. A two-class decoder always chooses a class on valid input; thresholding is **not a learned no-control detector**.

In [ ]:
import matplotlib.pyplot as plt
rows = clean["rows"]
t = [r["sample_time"] for r in rows]
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.step(t, [r["naive_cursor"] for r in rows], where="post", color="#D87C36", label="argmax cursor")
ax.step(t, [r["cursor"] for r in rows], where="post", color="#167D91", label="gated cursor")
ax.set(xlabel="Recorded sample time (s)", ylabel="Bounded virtual position", title="A simulated effector: displacement is not task success")
ax.legend(); plt.show()

In [ ]:
for row in [r for r in rows if "scores_left_right" in r][:5]:
    print({k: row[k] for k in ("sample_end", "scores_left_right", "status", "command", "audit_label")})
print(bci.save_bundle(clean))

## Read the metrics with their denominators
 `task_window_accuracy` uses valid windows wholly inside T1/T2 intervals. `conditional_task_command_accuracy` uses only emitted task commands; also report their count. Overlapping windows are not independent trials.
 Rest proxy rates divide commands in wholly T0 windows by **scored endpoint exposure**: the sum of their update durations. Report `scored_rest_seconds`, total annotated rest, and eligible rest windows together. Transitions, warm-up, and invalid intervals are excluded, not scored as successful inactivity.
 Instructed cued rest is not unrestricted everyday idle behavior. The recorded file has no ground-truth timestamp for a spontaneous decision. Processing time is measured locally, whereas packet age is simulated; neither measures physical feedback latency.

## Reflection
 A lower command count can reflect useful caution or excessive inhibition. What task-completion evidence would distinguish them? Why is a cursor at its limit not evidence that the participant controlled it successfully? What would you need to add for a real no-control detector?
 Continue with **03_faults_and_validation.ipynb**.